# Stage1 风险上限释放：原始整股可执行性预检
冻结来源和原生账户回读；不读取未来结果或拟合。


In [1]:
import copy, hashlib, json, os, sys
from pathlib import Path
repo_root=Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parents[1]
sys.path.insert(0,str(repo_root))
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.intraday_recovery_direct_allocation import project_legal_weights
from src.backtest.ibkr_tiered import TieredFees, settle_tiered_event

root=Path.cwd() if (Path.cwd()/'research/stage1_cap_release_execution_eligibility_20261023/plan.json').exists() else Path.cwd().parents[1]
card=root/'research/stage1_cap_release_execution_eligibility_20261023'; plan=json.loads((card/'plan.json').read_text()); src=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ['new_market_requests','new_model_fits','new_native_accounts','future_return_labels','new_bootstraps'])
for pathkey,hashkey in [('source_card','source_summary_sha256'),('training_tariff_plan','training_tariff_plan_sha256'),('previous_source_preflight','previous_source_preflight_sha256')]:
    p=root/src[pathkey]/'summary.json' if pathkey=='source_card' else root/src[pathkey]
    assert sha(p)==src[hashkey],pathkey
assert sha(root/src['source_card']/'plan.json')==src['source_plan_sha256']
for p,k in [('src/workflow/intraday_recovery_direct_allocation.py','projector_module_sha256'),('src/backtest/ibkr_tiered.py','tiered_module_sha256'),('src/backtest/stage1_ibkr.py','replay_module_sha256')]:assert sha(root/p)==src[k],p
old=json.loads((root/src['training_tariff_plan']).read_text()); source_summary=json.loads((root/src['source_card']/'summary.json').read_text())
assert source_summary['recorder_id']==src['source_recorder_id'] and source_summary['experiment_id']==src['source_experiment_id']
assert source_summary['nas_readback']['verified'] and source_summary['nas_readback']['sha256']==source_summary['nas_evidence']['sha256']
provider=(root/'data/qlib/full_path_tiered_allocation_20261013').resolve(); tracking=(root/src['source_card']/'qlib_recorder').resolve();os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(tracking),'default_exp_name':Path(src['source_card']).name}})
rec=R.get_recorder(experiment_name=Path(src['source_card']).name,recorder_id=src['source_recorder_id'])
frame=rec.load_object('inputs/source_frame'); bundle=rec.load_object('paths/reference_0'); action=rec.load_object('inputs/action_plan'); native=rec.load_object('native/reference_0')
assert native['audit']['status']=='passed' and native['audit']['slippage_bps']==0
path=bundle['replay'].reset_index(drop=True); targets=bundle['targets'].reset_index(drop=True); frame=frame.reset_index(drop=True)
assert len(frame)==len(path)==len(targets)==1061
assert frame.trade_date.equals(path.trade_date) and frame.trade_date.equals(targets.trade_date)
cal=list(pd.to_datetime(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day')).date)
assert cal==list(pd.to_datetime(frame.trade_date).dt.date)
spec=action['recovery_spec'];mincash=float(spec['minimum_cash_weight']);maxsoxl=float(spec['maximum_soxl_weight']);assert mincash==0.15 and maxsoxl==0.30
fees=TieredFees(old['tariff'],slippage_bps=0);q=np.zeros(2);cash=100000.;rows=[];max_control_target_error=0.;max_ledger_error=0.
for i in range(len(frame)):
    row=frame.iloc[i];saved=path.iloc[i];target=targets.iloc[i][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float)
    source_weights=row[['soxl_weight','soxx_weight']].to_numpy(float);cap=float(saved.current_beta_cap);raw_beta=float(3*source_weights[0]+source_weights[1]);gap=raw_beta-cap
    control_projection=project_legal_weights(*source_weights,cap,minimum_cash_weight=mincash,maximum_soxl_weight=maxsoxl)
    err=float(np.max(np.abs(control_projection-target)));max_control_target_error=max(max_control_target_error,err)
    assert err<1e-9,(i,err)
    signal_eligible=bool(gap>=0.10 and saved.execute_open_target)
    candidate_target=project_legal_weights(*source_weights,cap+0.10,minimum_cash_weight=mincash,maximum_soxl_weight=maxsoxl) if signal_eligible else target
    target_changed=bool(signal_eligible and np.max(np.abs(candidate_target-target))>1e-9)
    prices=row[['soxl_open','soxx_open']].to_numpy(float);raw_prices=row[['raw_soxl_open','raw_soxx_open']].to_numpy(float);stamp=pd.Timestamp(row.trade_date)+pd.Timedelta(hours=9,minutes=47)
    candidate=None
    if target_changed:
        candidate=settle_tiered_event(q,cash,desired=candidate_target,adjusted_prices=prices,raw_prices=raw_prices,stamp=stamp,fees=copy.deepcopy(fees))
    control=settle_tiered_event(q,cash,desired=target if bool(saved.execute_open_target) else None,adjusted_prices=prices,raw_prices=raw_prices,stamp=stamp,fees=fees)
    err=float(np.max(np.abs(np.r_[control['quantities'],control['cash']]-saved[['ending_soxl_quantity','ending_soxx_quantity','ending_cash']].to_numpy(float))))
    max_ledger_error=max(max_ledger_error,err);assert err<=1e-6,(i,err)
    if candidate is not None:
        nav=float(candidate['quantities']@prices+candidate['cash']);actual_cash_weight=candidate['cash']/nav
        changed_raw=bool(np.max(np.abs(candidate['quantities']-control['quantities'])*prices)/nav>1e-8)
        funded=bool(actual_cash_weight>=mincash-1e-6 and candidate['cash']>=-1e-7)
    else:changed_raw=False;funded=False
    rows.append({'year':pd.Timestamp(row.trade_date).year,'source_gap_010':gap>=.10,'control_execute':bool(saved.execute_open_target),'target_changed':target_changed,'whole_share_changed':changed_raw,'cash_floor_legal':funded,'executable':bool(target_changed and changed_raw and funded)})
    q,cash=control['quantities'],control['cash']
a=pd.DataFrame(rows);annual=[]
for year,g in a.groupby('year'):
    annual.append({'year':int(year),'sessions':len(g),'source_gap_010':int(g.source_gap_010.sum()),'control_execute_and_gap':int((g.source_gap_010 & g.control_execute).sum()),'projected_target_changed':int(g.target_changed.sum()),'whole_share_changed':int(g.whole_share_changed.sum()),'funded_cash_floor_executable':int(g.executable.sum())})
result={'decision':'physical_open_eligibility_only_no_outcome_label','source_recorder_id':rec.id,'sessions':len(a),'minimum_cash_weight':mincash,'maximum_soxl_weight':maxsoxl,'cap_release_beta':0.10,'max_control_target_parity_error':max_control_target_error,'max_reference_ledger_parity_error_usd':max_ledger_error,'source_gap_010_rows':int(a.source_gap_010.sum()),'control_execute_and_gap_rows':int((a.source_gap_010 & a.control_execute).sum()),'projected_target_changed_rows':int(a.target_changed.sum()),'whole_share_changed_rows':int(a.whole_share_changed.sum()),'funded_cash_floor_executable_rows':int(a.executable.sum()),'annual':annual,'new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'future_return_labels':0,'new_bootstraps':0,'limitation':'Candidate feasibility uses realized 09:47 source execution marks and is not a prediction-time eligibility mask. This checks one opening order only, not reset, outcome, risk feedback or complete policy. 2018-2022 training reference is not frozen 2023+ Stage1 full parent.'}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','source_recorder_id':rec.id,'source_NAS_sha256':source_summary['nas_evidence']['sha256'],'source_NAS_readback_verified':True,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps(result,ensure_ascii=False))

{"decision": "physical_open_eligibility_only_no_outcome_label", "source_recorder_id": "3663a1e073f042e383dbcf54969a9bc2", "sessions": 1061, "minimum_cash_weight": 0.15, "maximum_soxl_weight": 0.3, "cap_release_beta": 0.1, "max_control_target_parity_error": 0.0, "max_reference_ledger_parity_error_usd": 0.0, "source_gap_010_rows": 541, "control_execute_and_gap_rows": 541, "projected_target_changed_rows": 541, "whole_share_changed_rows": 541, "funded_cash_floor_executable_rows": 541, "annual": [{"year": 2018, "sessions": 53, "source_gap_010": 18, "control_execute_and_gap": 18, "projected_target_changed": 18, "whole_share_changed": 18, "funded_cash_floor_executable": 18}, {"year": 2019, "sessions": 252, "source_gap_010": 39, "control_execute_and_gap": 39, "projected_target_changed": 39, "whole_share_changed": 39, "funded_cash_floor_executable": 39}, {"year": 2020, "sessions": 253, "source_gap_010": 182, "control_execute_and_gap": 182, "projected_target_changed": 182, "whole_share_changed":

# Stage1 风险上限释放：原始整股可执行性

从保存的 2018–2022 Stage1 原架构 OOF 0bp Tiered 训练参考 Recorder，只读核对 1061 日的 Qlib 日历、来源、原控制投影目标以及原始整股/现金账本。原投影目标与复算逐日最大误差为 0，控制账本的原始数量和现金误差为 0。

按冻结的额外 0.10 beta 上限，用原 `project_legal_weights`、15% 最低现金和 30% 最高 SOXL 权重重投影。**541 日**原始目标有至少 0.10 的上限缺口；这 541 日原控制都执行，候选投影目标改变，真实整股数量改变，且候选订单扣除 IBKR Tiered 费用后的现金仍满足 15% 下限。逐年可行动日为 2018:18、2019:39、2020:182、2021:56、2022:246；2019/2021 支持较薄。

本卡只检查同一开盘的机械可执行性。09:47 执行价格是在 09:46 决策后观察的，不能被当作决策时筛选信号；尚未计算下一日收益、共同复位标签、风险状态反馈、完整账户或 SOXX 超额。下一步需要另行冻结并计算同状态、共同复位、两条独立月度费率计数器的局部费用后价值，再先过跨年信息门。
